# LIBRARIES

In [8]:
import random
from queue import Queue, PriorityQueue
import scipy.stats as st
import numpy as np


# CLASSES

## MEASURES

In [10]:
class Measure:
    def __init__(self,Narr,Ndep,NAveraegUser,OldTimeEvent,AverageDelay):
        self.arr = Narr
        self.dep = Ndep
        self.ut = NAveraegUser
        self.oldT = OldTimeEvent
        self.delay = AverageDelay

## CLIENT

In [11]:
class Client:
    def __init__(self,type,arrival_time):
        self.type = type
        self.arrival_time = arrival_time

## SERVER

In [12]:
class Server(object):

    # constructor
    def __init__(self):

        # whether the server is idle or not
        self.idle = True

# CONSTANT

In [24]:

SERVICE = 10.0 # SERVICE is the average service time; service rate = 1/SERVICE
#ARRIVAL = 5.0  ARRIVAL is the average inter-arrival time; arrival rate = 1/ARRIVAL
ARRIVALS = [3*i for i in range(1,5)]
LOAD=[SERVICE/arrival for arrival in ARRIVALS] # This relationship holds for M/M/1

TYPE1 = 1 

SIM_TIME = 500000

arrivals=0
users=0
BusyServer=False # True: server is currently busy; False: server is currently idle

MM1=[]

# FUNCTIONS

## ARRIVAL

In [13]:
def arrival(data,time, FES, queue, arr_time):
    global users
    
    #print("Arrival no. ",data.arr+1," at time ",time," with ",users," users" )
    
    # cumulate statistics
    data.arr += 1
    data.ut += users*(time-data.oldT)
    data.oldT = time

    # sample the time until the next event
    inter_arrival = random.expovariate(lambd=1.0/arr_time)
        
    # schedule the next arrival
    FES.put((time + inter_arrival, "arrival"))

    users += 1
    
    # create a record for the client
    client = Client(TYPE1,time)

    # insert the record in the queue
    queue.append(client)

    # if the server is idle start the service
    if users==1:
        
        # sample the service time
        service_time = random.expovariate(1.0/SERVICE)
        #service_time = 1 + random.uniform(0, SEVICE_TIME)

        # schedule when the client will finish the server
        FES.put((time + service_time, "departure"))

## DEPARTURES

In [14]:
def departure(data,time, FES, queue):
    global users

    #print("Departure no. ",data.dep+1," at time ",time," with ",users," users" )
        
    # cumulate statistics
    data.dep += 1
    data.ut += users*(time-data.oldT)
    data.oldT = time
    
    # get the first element from the queue
    client = queue.pop(0)
    
    # do whatever we need to do when clients go away
    
    data.delay += (time-client.arrival_time)
    users -= 1
    
    # see whether there are more clients to in the line
    if users >0:
        # sample the service time
        service_time = random.expovariate(1.0/SERVICE)

        # schedule when the client will finish the server
        FES.put((time + service_time, "departure"))


## CONFIDENCE INTERVALS

In [15]:
def confidence_interval(data, confidence=0.95):
    
    n = len(data)
    mean = np.mean(data)
    std_err = st.sem(data)
    h = std_err * st.t.ppf((1 + confidence) / 2, n - 1)

    return mean, mean - h, mean + h

# MAIN

In [27]:

# Run 25 simulations for each scenario (ARRIVAL value) and collect results
num_simulations = 25
all_results = {}

for arrival_scenario in ARRIVALS:
    results = {
        'arrivals': [],
        'departures': [],
        'avg_users': [],
        'avg_delay': [],
        'arrival_rate': [],
        'departure_rate': []
    }
    
    for sim in range(num_simulations):
        # Reset random seed for reproducibility
        random.seed(42 + sim)
        
        # Reset global variables for this simulation
        users = 0
        data = Measure(0, 0, 0, 0, 0)
        MM1 = []
        
        # the simulation time 
        time = 0
        
        # the list of events in the form: (time, type)
        FES = PriorityQueue()
        
        # schedule the first arrival at t=0
        FES.put((0, "arrival"))
    
        
        # simulate until the simulated time reaches a constant
        while time < SIM_TIME:
            (time, event_type) = FES.get()
        
            if event_type == "arrival":
                arrival(data, time, FES, MM1, arrival_scenario)
        
            elif event_type == "departure":
                departure(data,time, FES, MM1)
        
        
        # Collect results from this simulation
        results['arrivals'].append(data.arr)
        results['departures'].append(data.dep)
        results['avg_users'].append(data.ut / time)
        results['avg_delay'].append(data.delay / data.dep if data.dep > 0 else 0)
        results['arrival_rate'].append(data.arr / time)
        results['departure_rate'].append(data.dep / time)
    
    all_results[arrival_scenario] = results

# Print summary statistics for each scenario
for arrival_scenario in ARRIVALS:
    results = all_results[arrival_scenario]
    
    print("=" * 60)
    print(f"RESULTS FROM 25 SIMULATIONS - ARRIVAL = {arrival_scenario}")
    print("=" * 60)
    
    print("\nNUMBER OF ARRIVALS:")
    mean, lower, upper = confidence_interval(results['arrivals'])
    print(f"  Mean: {mean:.2f}, CI[95%]: [{lower:.2f}, {upper:.2f}]")
    
    print("\nNUMBER OF DEPARTURES:")
    mean, lower, upper = confidence_interval(results['departures'])
    print(f"  Mean: {mean:.2f}, CI[95%]: [{lower:.2f}, {upper:.2f}]")
    
    print("\nAVERAGE NUMBER OF USERS IN SYSTEM:")
    mean, lower, upper = confidence_interval(results['avg_users'])
    print(f"  Mean: {mean:.4f}, CI[95%]: [{lower:.4f}, {upper:.4f}]")
    
    print("\nAVERAGE DELAY (per customer):")
    mean, lower, upper = confidence_interval(results['avg_delay'])
    print(f"  Mean: {mean:.4f}, CI[95%]: [{lower:.4f}, {upper:.4f}]")
    
    print("\nARRIVAL RATE:")
    mean, lower, upper = confidence_interval(results['arrival_rate'])
    print(f"  Mean: {mean:.6f}, CI[95%]: [{lower:.6f}, {upper:.6f}]")
    
    print("\nDEPARTURE RATE:")
    mean, lower, upper = confidence_interval(results['departure_rate'])
    print(f"  Mean: {mean:.6f}, CI[95%]: [{lower:.6f}, {upper:.6f}]")
    
    print(f"\nLOAD FACTOR (λ/μ) = SERVICE/ARRIVAL = {SERVICE/arrival_scenario:.4f}")
    
    print()


RESULTS FROM 25 SIMULATIONS - ARRIVAL = 3

NUMBER OF ARRIVALS:
  Mean: 166513.28, CI[95%]: [166299.74, 166726.82]

NUMBER OF DEPARTURES:
  Mean: 49982.72, CI[95%]: [49888.44, 50077.00]

AVERAGE NUMBER OF USERS IN SYSTEM:
  Mean: 58232.7782, CI[95%]: [58115.6100, 58349.9464]

AVERAGE DELAY (per customer):
  Mean: 174796.9207, CI[95%]: [174502.6449, 175091.1964]

ARRIVAL RATE:
  Mean: 0.333025, CI[95%]: [0.332598, 0.333452]

DEPARTURE RATE:
  Mean: 0.099965, CI[95%]: [0.099776, 0.100154]

LOAD FACTOR (λ/μ) = SERVICE/ARRIVAL = 3.3333

RESULTS FROM 25 SIMULATIONS - ARRIVAL = 6

NUMBER OF ARRIVALS:
  Mean: 83256.84, CI[95%]: [83139.05, 83374.63]

NUMBER OF DEPARTURES:
  Mean: 49952.16, CI[95%]: [49850.96, 50053.36]

AVERAGE NUMBER OF USERS IN SYSTEM:
  Mean: 16666.2682, CI[95%]: [16584.8860, 16747.6504]

AVERAGE DELAY (per customer):
  Mean: 100175.5468, CI[95%]: [99675.5300, 100675.5636]

ARRIVAL RATE:
  Mean: 0.166513, CI[95%]: [0.166277, 0.166748]

DEPARTURE RATE:
  Mean: 0.099904, CI[95